# Fleet Management Data Collection

If you've run through the road following, your should be familiar following three steps

1.  Data collection
2.  Training
3.  Deployment

In this notebook, we'll do the same exact thing!  Except, instead of classification, you'll learn a different fundamental technique, **object detection**, that we'll use to
enable JetBot to follow an object (or really, any object such as jetbot).

1. Place the JetBot in different positions on a path (offset from center, different angles, etc)

>  Remember from road following, data variation is key!

2. Display the live camera feed from the robot
3. Using a gamepad controller, place a 'green dot', which corresponds to the left top point and right top point that can cover the boundary Jetbot, on the image.
4. Store as the label value, the left top point(x1, y1) and right top point (x2, y2) values, of this green dot along with the image from the Jetbot's camera, which is the bounding box value of the Jetbot

Then, in the training notebook, we'll train a neural network to predict the label values of our label.  In the live demo, we'll use
the predicted bounding value (x1, y1) and (x2, y2) to compute an approximate size value and position od Jetbot.

So how do you decide exactly where to place the target for this example?  Here is a guide we think may help

1.  Look at the live video feed from the camera
2.  Place a target Jetbot where it could stay.


Assuming our deep learning model works as intended, these labeling guidelines should ensure the robot can be accurately detected


### Import Libraries

So lets get started by importing all the required libraries for "data collection" purpose. We will mainly use OpenCV to visualize and save image with labels. Libraries such as uuid, datetime are used for image naming. 

In [ ]:
# IPython Libraries for display and widgets
import ipywidgets
import traitlets
import ipywidgets.widgets as widgets
from IPython.display import display

# Camera and Motor Interface for JetBot
from jetbot import Robot, Camera, bgr8_to_jpeg

# Basic Python packages for image annotation
from uuid import uuid1
import os
import json
import glob
import datetime
import numpy as np
import cv2
import time

### Data Collection

Let's display our camera like we did in the teleoperation notebook, however this time with using a special ipywidget called `jupyter_clickable_image_widget` that lets you click on the image and take the coordinates for data annotation.
This eliminates the needs of using the gamepad for data annotation.

We use Camera Class from JetBot to enable CSI MIPI camera. Our neural network takes a 224x224 pixel image as input. We'll set our camera to that size to minimize the filesize of our dataset (we've tested that it works for this task). In some scenarios it may be better to collect data in a larger image size and downscale to the desired size later.

The following block of code will display the live image feed for you to click on for annotation on the left, as well as the snapshot of last annotated image (with a green circle showing where you clicked) on the right.
Below it shows the number of images we've saved.  

When you click on the left live image, it stores a file in the ``dataset_xy`` folder with files named

``xy_<x value>_<y value>_<uuid>.jpg``

When we train, we load the images and parse the x, y values from the filename.
Here `<x value>` and `<y value>` are the coordinates **in pixels** (count from the top left corner).



In [ ]:
from jupyter_clickable_image_widget import ClickableImageWidget

DATASET_DIR = 'jetbot_bbox'

# we have this "try/except" statement because these next functions can throw an error if the directories exist already
try:
    os.makedirs(DATASET_DIR)
except FileExistsError:
    print('Directories not created because they already exist')

camera = Camera()

# create image preview
camera_widget = ClickableImageWidget(width=camera.width, height=camera.height)
snapshot_widget = ipywidgets.Image(width=camera.width, height=camera.height)
traitlets.dlink((camera, 'value'), (camera_widget, 'value'), transform=bgr8_to_jpeg)

# create widgets
count_widget = ipywidgets.IntText(description='count')
# manually update counts at initialization
count_widget.value = len(glob.glob(os.path.join(DATASET_DIR, '*.jpg')))


# display buttons for start and stop running
button_layout = widgets.Layout(width='150px', height='40px', align_self='center')
redo_button = widgets.Button(description='Stop', tooltip='Click to stop running', icon='stop', layout=button_layout)
redo_button.style.button_color='Red'
save_button = widgets.Button(description='Start', tooltip='Click to start running', icon='play', layout=button_layout)
save_button.style.button_color='lightBlue'

button_box = widgets.HBox([save_button, redo_button], layout=widgets.Layout(justify_content='space-around', width='30%'))
save_button.disabled=True
redo_button.disabled=True

count_xy = 2
bbox = []

def save_snapshot_bbox(_, content, msg):
    global count_xy, bbox
    if count_xy==0:
        return

    if content['event'] == 'click':
        data = content['eventData']
        x = data['offsetX']    # left top x
        y = data['offsetY']    # left top y
        snapshot = camera.value.copy()
        cv2.circle(snapshot, (x, y), 8, (0, 255, 0), 3)
        count_xy-=1
        bbox.append((x, y))

    if count_xy==0:
        save_button.disabled=False
        redo_button.disabled=False
        snapshot = camera.value.copy()
        cv2.rectangle(snapshot, bbox[0], bbox[1], (0, 255, 0), 3)

def save_bbox(change):
    global count_xy, bbox
    # save to disk
    x1 = bbox[0][0]
    y1 = bbox[0][1]
    x2 = bbox[1][0]
    y2 = bbox[1][1]
    uuid = f'bbox_{x1}_{y1}_{x2}_{y2}_{uuid1()}'
    image_path = os.path.join(DATASET_DIR, uuid + '.jpg')
    with open(image_path, 'wb') as f:
        f.write(camera_widget.value)

    # display saved snapshot
    snapshot = camera.value.copy()
    snapshot = cv2.rectangle(snapshot, (x1, y1), (x2, y2), (0, 255, 0), 3)
    snapshot_widget.value = bgr8_to_jpeg(snapshot)
    count_widget.value = len(glob.glob(os.path.join(DATASET_DIR, '*.jpg')))
    # fetch next bbox
    bbox = []
    count_xy = 2
    save_button.disabled=True
    redo_button.disabled=True

def redo(change):
    global count_xy, bbox
    count_xy = 2
    bbox = []
    save_button.disabled=True
    redo_button.disabled=True

camera_widget.on_msg(save_snapshot_bbox)

save_button.on_click(save_bbox)
redo_button.on_click(redo)

data_collection_widget = ipywidgets.VBox([
    ipywidgets.HBox([camera_widget, snapshot_widget]),
    count_widget
])

display(data_collection_widget)
display(button_box)

Again, let's close the camera conneciton properly so that we can use the camera in other notebooks.

In [ ]:
camera.stop()

### Next

Once you've collected enough data, we'll need to copy that data to our GPU desktop or cloud machine for training. First, we can call the following terminal command to compress our dataset folder into a single zip file.  

> If you're training on the JetBot itself, you can skip this step!

The ! prefix indicates that we want to run the cell as a shell (or terminal) command.

The -r flag in the zip command below indicates recursive so that we include all nested files, the -q flag indicates quiet so that the zip command doesn't print any output

In [ ]:
def timestr():
    return str(datetime.datetime.now().strftime('%Y-%m-%d_%H-%M-%S'))

!zip -r -q Jetbot_bbox_{DATASET_DIR}_{timestr()}.zip {DATASET_DIR}

You should see a file named road_following_<Date&Time>.zip in the Jupyter Lab file browser. You should download the zip file using the Jupyter Lab file browser by right clicking and selecting Download.